# 10 - v0.1 Onboarding Journey

This notebook is the runnable version of the v0.1 onboarding probe. It uses only the kernel package surface and the native reasoning path:

1. Define a small schema with primary and secondary identity fields.
2. Write data with `sdk.ref`, `sdk.set`, and `sdk.add`.
3. Read with `sdk.get` and `sdk.run(Query(...))`.
4. Derive and accept a native candidate.
5. Export a kernel audit package.
6. Read the package back with `kernel.audit` and inspect the candidate evidence tree.

No `agent`, `service`, `domains`, PyReason, ProbLog, or external API key is required.

## 0. Imports

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

# Works both when launched from repo root and from examples/.
cwd = Path.cwd().resolve()
repo_src = cwd / "src"
if not repo_src.exists() and cwd.name == "examples":
    repo_src = cwd.parent / "src"
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

In [ ]:
from kernel.adapters.souffle.package import ExportOptions
from kernel.sdk import Derivation, Entity, Field, Identity, Pred, Query, SDKStore, vars as sdk_vars

## 1. Define the schema

`User.user_id` is the primary identity. `User.locale` is a secondary identity coordinate with a default value. `name` is single-valued, `tag` is multi-valued, and `home` is an entity reference.

In [ ]:
class Country(Entity):
    code: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")


class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity(default="zh")
    name: str = Field(cardinality="single")
    tag: str = Field(cardinality="multi")
    home: Country = Field(cardinality="single")

## 2. Create refs and write facts

The v0.1 hardened path lets `sdk.set` and `sdk.add` materialize the entity identity and `<T>:exists` facts through the application write-plan path. The user does not need to start with `sdk.batch`.

In [ ]:
sdk = SDKStore([Country, User])

de = sdk.ref(Country, code="DE")
alice = sdk.ref(User, user_id="u1", locale="zh")

sdk.set(Country.name, de, "Germany")
sdk.set(User.name, alice, "Alice")
sdk.set(User.home, alice, de)
sdk.add(User.tag, alice, "admin")
sdk.add(User.tag, alice, "vip")

print(alice)

## 3. Read the entity snapshot

In [ ]:
snapshot = sdk.get(User, user_id="u1", locale="zh")

print(snapshot.name)
print(sorted(snapshot.tag))
print(snapshot.home)

## 4. Query with the SDK DSL

In [ ]:
with sdk_vars("u", "name") as (u, name):
    q = Query(
        head=[User(u), User.name(value=name)],
        where=[User(u), Pred("user:name", u, name)],
    )

rows = sdk.run(q)
rows

## 5. Derive and accept a native candidate

This native derivation turns an `admin` tag into a derived `audited` tag. It demonstrates that facts written through `sdk.set/add` are visible to the reasoning path.

In [ ]:
with sdk_vars("u", "loc", "derived") as (u, loc, derived):
    derivation = Derivation(
        id="journey.derived_tag",
        version="1.0.0",
        where=[User(u), u.locale == loc, Pred("user:tag", u, "admin"), derived == "audited"],
        head=User.tag(locale=loc, tag=derived),
    )

candidates = sdk.evaluate(derivation, mode="native")
print(len(candidates))

accepted = sdk.accept(candidates[0], approved_by="journey", note="accept derived audit tag")
accepted

In [ ]:
with sdk_vars("u") as (u,):
    derived_query = Query(
        head=User(u),
        where=[Pred("user:tag", u, "audited")],
    )

derived_rows = sdk.run(derived_query)
derived_rows

## 6. Export a kernel audit package

This verifies the kernel audit package boundary. It does not render a static site; rendered audit pages are a separate `service.static_ui` / monorepo delivery surface.

In [ ]:
package_dir = Path(tempfile.mkdtemp(prefix="factpy_audit_package_"))
sdk.export_package(package_dir, ExportOptions(package_kind="audit"))

print(package_dir)
print((package_dir / "manifest.json").exists())
print(sorted(p.name for p in package_dir.iterdir()))

## 7. Read the audit package back

Export is only half of the delivery story. The same kernel wheel can also read the package back and expose run, candidate, and accept-write ledgers.

In [ ]:
from kernel.audit import (
    AuditQuery,
    build_candidate_evidence_tree_dto,
    build_candidate_evidence_tree_narrative_dto,
    build_candidate_evidence_tree_summary_dto,
    load_audit_package,
    render_evidence_graph_html,
)
from IPython.display import HTML, display

In [ ]:
package = load_audit_package(package_dir)
audit = AuditQuery(package)

runs = audit.list_runs()
accepted_candidates = audit.list_candidates(state="accepted")
accept_writes = audit.list_accept_writes()

runs_view = [
    {
        "run_id": row["run_id"],
        "candidate_count": len(row.get("candidate_ids", [])),
        "decision_count": row.get("decision_count", 0),
        "has_failures": row.get("has_failures", False),
    }
    for row in runs
]

candidates_view = [
    {
        "candidate_id": row["candidate_id"],
        "state": row["state"],
        "pred_id": row["pred_id"],
        "support_kind": row["support_kind"],
    }
    for row in accepted_candidates
]

accept_writes_view = [
    {
        "candidate_id": row["candidate_id"],
        "asrt_id": row["asrt_id"],
        "pred_id": row["pred_id"],
        "approved_by": row.get("approved_by"),
    }
    for row in accept_writes
]

print("runs")
print(runs_view)
print("accepted candidates")
print(candidates_view)
print("accept writes")
print(accept_writes_view)

candidate_id = accepted_candidates[0]["candidate_id"]
candidate_id

## 8. Inspect the candidate evidence tree

For native derivations, the audit package stores enough witness information to build a candidate evidence tree, summarize it, and render a compact narrative. Some candidates may also carry a materialized `EvidenceGraph`; this native path does not require one.

In [ ]:
raw_tree = build_candidate_evidence_tree_dto(audit, candidate_id)
summary_dto = build_candidate_evidence_tree_summary_dto(audit, candidate_id)
narrative_dto = build_candidate_evidence_tree_narrative_dto(audit, candidate_id)

summary = summary_dto["summary"]
narrative = narrative_dto["narrative"]

print(raw_tree["root"]["title"])
print(summary)
print(narrative["headline"])
print(narrative["evidence_lines"])

In [ ]:
graph = audit.get_candidate_evidence_graph(candidate_id)

if graph is None:
    print("No materialized EvidenceGraph for this native candidate; use the raw tree, summary, and narrative DTOs above.")
else:
    display(HTML(render_evidence_graph_html(graph)))